# 4장. 딥러닝 연습 (텐서 · 역전파 · 학습 루프 · 과적합 · Attention)

CPU만으로 몇 초 안에 끝나게 작은 데이터로 연습해요. GPU가 있으면 `device`만 바꾸면 돼요.

> ★ 이 장의 목표: **학습 루프 5줄**(순전파 → 손실 → `zero_grad` → `backward` → `step`)을 안 보고 쓸 수 있게

In [ ]:
# ▶ 먼저 이 셀을 실행하세요 (채점 도우미)
import os
STRICT = os.environ.get("PRACTICE_STRICT") == "1"
___ = None  # 빈칸 표시. 이 자리를 채우면 돼요

def check(name, cond, hint=""):
    """cond가 참이면 ✅, 아니면 ❌ (cond는 값이나 lambda)"""
    try:
        ok = bool(cond() if callable(cond) else cond)
    except Exception as e:
        ok, hint = False, f"{type(e).__name__}: {e}"
    print(("✅ " if ok else "❌ ") + name + ("" if ok or not hint else f"  → {hint}"))
    if STRICT and not ok:
        raise AssertionError(name)

In [ ]:
import math
import torch, torch.nn as nn, torch.nn.functional as F
torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print(torch.__version__, "| device:", device)

## 4-1. 텐서와 모양(shape) — 디버깅의 절반
`nn.Linear(4, 16)`은 `y = x @ W.T + b` 이고, `W`의 모양은 `(16, 4)`, `b`는 `(16,)`예요.

In [ ]:
x = torch.randn(32, 4)                 # 배치 32개, 특성 4개
layer = nn.Linear(4, 16)

shape_answers = {
    "layer(x).shape":       ___,       # torch.Size([..., ...]) 또는 튜플로
    "layer.weight.shape":   ___,
    "layer.bias.shape":     ___,
    "x.view(32, 2, 2).shape": ___,
    "x.T.shape":            ___,
}
manual = ___                           # layer를 쓰지 말고 weight, bias로 직접 계산

<details><summary>💡 정답 보기 (먼저 직접 풀어보고 열기)</summary>

```python
x = torch.randn(32, 4)
layer = nn.Linear(4, 16)

shape_answers = {
    "layer(x).shape":       (32, 16),
    "layer.weight.shape":   (16, 4),
    "layer.bias.shape":     (16,),
    "x.view(32, 2, 2).shape": (32, 2, 2),
    "x.T.shape":            (4, 32),
}
manual = x @ layer.weight.T + layer.bias
```
</details>

In [ ]:
real = [layer(x).shape, layer.weight.shape, layer.bias.shape, x.view(32, 2, 2).shape, x.T.shape]
check("shape 예상", lambda: [tuple(s) for s in shape_answers.values()] == [tuple(s) for s in real])
check("Linear = x @ W.T + b", lambda: torch.allclose(manual, layer(x), atol=1e-6))

## 4-2. 활성화 함수가 없으면?
Linear 두 층을 활성화 없이 쌓으면 **Linear 한 층과 똑같아요.** 직접 증명해 봐요.
`l2(l1(x)) = (x @ W1.T + b1) @ W2.T + b2` 를 정리해서 **하나의** `W`, `b`로 만들어요.

In [ ]:
l1, l2 = nn.Linear(4, 16), nn.Linear(16, 2)
W1, b1, W2, b2 = l1.weight, l1.bias, l2.weight, l2.bias

W = ___          # 모양 (2, 4)
b = ___          # 모양 (2,)
one_layer = x @ W.T + b

<details><summary>💡 정답 보기 (먼저 직접 풀어보고 열기)</summary>

```python
l1, l2 = nn.Linear(4, 16), nn.Linear(16, 2)
W1, b1, W2, b2 = l1.weight, l1.bias, l2.weight, l2.bias

W = W2 @ W1
b = W2 @ b1 + b2
one_layer = x @ W.T + b
```
</details>

In [ ]:
with torch.no_grad():
    check("두 층 = 한 층 (활성화 없을 때)", lambda: torch.allclose(l2(l1(x)), one_layer, atol=1e-5))
    check("ReLU를 넣으면 더 이상 같지 않다", lambda: not torch.allclose(l2(F.relu(l1(x))), one_layer, atol=1e-3))

## 4-3. 손실 함수 직접 계산
- 분류: **CrossEntropy** = `-log(softmax(logits)[정답])`의 평균
- 회귀: **MSE** = `(예측 - 정답)²`의 평균

In [ ]:
logits = torch.tensor([[2.0, 0.5, -1.0],
                       [0.1, 0.2,  3.0]])
target = torch.tensor([0, 2])

probs = ___                                    # 행마다 softmax
ce_manual = ___                                # 정답 칸의 -log 확률 평균

pred, true = torch.tensor([2.5, 0.0, 2.0]), torch.tensor([3.0, -0.5, 2.0])
mse_manual = ___

<details><summary>💡 정답 보기 (먼저 직접 풀어보고 열기)</summary>

```python
logits = torch.tensor([[2.0, 0.5, -1.0],
                       [0.1, 0.2,  3.0]])
target = torch.tensor([0, 2])

probs = logits.exp() / logits.exp().sum(dim=1, keepdim=True)
ce_manual = -probs[torch.arange(len(target)), target].log().mean()

pred, true = torch.tensor([2.5, 0.0, 2.0]), torch.tensor([3.0, -0.5, 2.0])
mse_manual = ((pred - true) ** 2).mean()
```
</details>

In [ ]:
check("softmax 행 합 = 1", lambda: torch.allclose(probs.sum(1), torch.ones(2)))
check("CrossEntropy", lambda: torch.isclose(ce_manual, nn.CrossEntropyLoss()(logits, target)))
check("MSE", lambda: torch.isclose(mse_manual, nn.MSELoss()(pred, true)))

## 4-4. 역전파 = 연쇄법칙 자동 계산
`y = (3w + 1)²` 에서 `w = 2`일 때 `dy/dw`를 **손으로** 계산해 적고, `autograd`와 비교해요.
(겉미분 × 속미분: `2(3w+1) × 3`)

In [ ]:
w = torch.tensor(2.0, requires_grad=True)
y = (3 * w + 1) ** 2
y.backward()

my_grad = ___            # 손으로 계산한 숫자
print("autograd:", w.grad.item())

<details><summary>💡 정답 보기 (먼저 직접 풀어보고 열기)</summary>

```python
w = torch.tensor(2.0, requires_grad=True)
y = (3 * w + 1) ** 2
y.backward()

my_grad = 2 * (3 * 2 + 1) * 3     # = 42
print("autograd:", w.grad.item())
```
</details>

In [ ]:
check("연쇄법칙", lambda: my_grad == w.grad.item() == 42)

## 4-5. 경사하강법을 손으로 (옵티마이저 없이)
`y = 3x + 2` 를 따르는 데이터에서 `w`, `b`를 배워요. 기울기 **반대 방향**으로 `lr`만큼 이동해요.
`torch.no_grad()` 안에서 값을 고치고, 다음 반복을 위해 기울기를 0으로 지워요.

In [ ]:
X1 = torch.linspace(-1, 1, 50)
Y1 = 3 * X1 + 2
w = torch.zeros(1, requires_grad=True)
b = torch.zeros(1, requires_grad=True)
lr = 0.1

for step in range(300):
    loss = ((w * X1 + b - Y1) ** 2).mean()
    loss.backward()
    with torch.no_grad():
        w -= ___
        b -= ___
        w.grad.zero_(); b.grad.zero_()
print(f"w={w.item():.3f}, b={b.item():.3f}, loss={loss.item():.5f}")

<details><summary>💡 정답 보기 (먼저 직접 풀어보고 열기)</summary>

```python
X1 = torch.linspace(-1, 1, 50)
Y1 = 3 * X1 + 2
w = torch.zeros(1, requires_grad=True)
b = torch.zeros(1, requires_grad=True)
lr = 0.1

for step in range(300):
    loss = ((w * X1 + b - Y1) ** 2).mean()
    loss.backward()
    with torch.no_grad():
        w -= lr * w.grad
        b -= lr * b.grad
        w.grad.zero_(); b.grad.zero_()
print(f"w={w.item():.3f}, b={b.item():.3f}, loss={loss.item():.5f}")
```
</details>

In [ ]:
check("w ≈ 3", lambda: abs(w.item() - 3) < 0.05)
check("b ≈ 2", lambda: abs(b.item() - 2) < 0.05)

### 학습률(lr) 실험
`f(w) = w²`의 최솟값(0)을 찾아요. 기울기는 `2w`라서 한 번 갱신하면 `w ← w - lr·2w = (1 - 2lr)·w` 예요.
아래 lr 후보 각각이 **"수렴"**, **"발산"**, **"너무 느림"** 중 무엇인지 먼저 예상하고, 실험으로 확인해요.

In [ ]:
lr_guess = {0.001: ___, 0.3: ___, 1.1: ___}

<details><summary>💡 정답 보기 (먼저 직접 풀어보고 열기)</summary>

```python
lr_guess = {0.001: "너무 느림", 0.3: "수렴", 1.1: "발산"}   # |1-2lr| 가 1보다 크면 발산
```
</details>

In [ ]:
def run(lr, steps=50):
    w = 5.0
    for _ in range(steps):
        w -= lr * 2 * w
    return w

def label(w_final):
    return "발산" if abs(w_final) > 5 else ("수렴" if abs(w_final) < 0.01 else "너무 느림")

actual_lr = {lr: label(run(lr)) for lr in lr_guess}
print({lr: round(run(lr), 4) for lr in lr_guess})
check("학습률 예상", lambda: lr_guess == actual_lr, str(actual_lr))

## 4-6. ★ PyTorch 학습 루프 (외우기)
PDF의 5줄을 채워요. `model.train()` / `model.eval()` 과 `torch.no_grad()`도 잊지 말기.

In [ ]:
torch.manual_seed(0)
X = torch.randn(200, 4)
y = (X[:, 0] + X[:, 1] > 0).long()                  # 가짜 2분류 데이터

model = nn.Sequential(nn.Linear(4, 16), nn.ReLU(), nn.Linear(16, 2))
loss_fn = nn.CrossEntropyLoss()
opt = torch.optim.Adam(model.parameters(), lr=0.01)

for epoch in range(100):
    model.train()
    logits = ___        # 1) 순전파
    loss = ___          # 2) 손실
    ___                 # 3) 기울기 초기화
    ___                 # 4) 역전파
    ___                 # 5) 가중치 갱신

model.eval()
with torch.no_grad():
    acc = (model(X).argmax(1) == y).float().mean().item()
print(f"loss={loss.item():.3f} acc={acc:.2f}")

<details><summary>💡 정답 보기 (먼저 직접 풀어보고 열기)</summary>

```python
torch.manual_seed(0)
X = torch.randn(200, 4)
y = (X[:, 0] + X[:, 1] > 0).long()

model = nn.Sequential(nn.Linear(4, 16), nn.ReLU(), nn.Linear(16, 2))
loss_fn = nn.CrossEntropyLoss()
opt = torch.optim.Adam(model.parameters(), lr=0.01)

for epoch in range(100):
    model.train()
    logits = model(X)
    loss = loss_fn(logits, y)
    opt.zero_grad()
    loss.backward()
    opt.step()

model.eval()
with torch.no_grad():
    acc = (model(X).argmax(1) == y).float().mean().item()
print(f"loss={loss.item():.3f} acc={acc:.2f}")
```
</details>

In [ ]:
check("정확도 95% 이상", lambda: acc >= 0.95)
check("loss 0.1 미만", lambda: loss.item() < 0.1)

### `zero_grad()`를 빼면?
`backward()`는 기울기를 **더해요(누적)**. 같은 배치로 두 번 `backward()`만 하면 기울기가 몇 배가 되는지 예상해요.

In [ ]:
times = ___      # 몇 배?

<details><summary>💡 정답 보기 (먼저 직접 풀어보고 열기)</summary>

```python
times = 2        # 누적되니까 2배 → 지우지 않으면 엉뚱한 방향으로 크게 움직임
```
</details>

In [ ]:
m = nn.Linear(4, 2)
out = lambda: loss_fn(m(X), y)
out().backward(); g1 = m.weight.grad.clone()
out().backward(); g2 = m.weight.grad.clone()       # zero_grad 없이 한 번 더
check("기울기 누적 배수", lambda: torch.allclose(g2, g1 * times))

## 4-7. 과적합 알아채기 + 조기 종료
데이터가 적고(60개) 라벨에 잡음이 있는데 모델이 크면, **train 손실은 계속 ↓, valid 손실은 어느 순간 ↑** 해요.
루프 안에서 valid 손실이 **가장 낮았던** 에폭과 그때의 가중치를 저장해요 (조기 종료의 핵심).

In [ ]:
import copy
torch.manual_seed(1)
def make(n):
    X = torch.randn(n, 10)
    y = (X[:, 0] > 0).long()
    flip = torch.rand(n) < 0.2                     # 라벨 20%는 잡음
    y[flip] = 1 - y[flip]
    return X, y
Xtr, ytr = make(60)
Xva, yva = make(500)

big = nn.Sequential(nn.Linear(10, 256), nn.ReLU(), nn.Linear(256, 256), nn.ReLU(), nn.Linear(256, 2))
opt = torch.optim.Adam(big.parameters(), lr=0.01)
history, best_val, best_epoch, best_state = [], float("inf"), -1, None

for epoch in range(300):
    big.train()
    loss = loss_fn(big(Xtr), ytr)
    opt.zero_grad(); loss.backward(); opt.step()

    big.eval()
    with torch.no_grad():
        val = loss_fn(big(Xva), yva).item()
    history.append((loss.item(), val))
    if ___:                                         # valid 손실이 지금까지 최저면
        best_val, best_epoch = ___, ___
        best_state = copy.deepcopy(big.state_dict())

print(f"최저 valid {best_val:.3f} @ epoch {best_epoch} / 마지막: train {history[-1][0]:.3f}, valid {history[-1][1]:.3f}")

<details><summary>💡 정답 보기 (먼저 직접 풀어보고 열기)</summary>

```python
import copy
torch.manual_seed(1)
def make(n):
    X = torch.randn(n, 10)
    y = (X[:, 0] > 0).long()
    flip = torch.rand(n) < 0.2
    y[flip] = 1 - y[flip]
    return X, y
Xtr, ytr = make(60)
Xva, yva = make(500)

big = nn.Sequential(nn.Linear(10, 256), nn.ReLU(), nn.Linear(256, 256), nn.ReLU(), nn.Linear(256, 2))
opt = torch.optim.Adam(big.parameters(), lr=0.01)
history, best_val, best_epoch, best_state = [], float("inf"), -1, None

for epoch in range(300):
    big.train()
    loss = loss_fn(big(Xtr), ytr)
    opt.zero_grad(); loss.backward(); opt.step()

    big.eval()
    with torch.no_grad():
        val = loss_fn(big(Xva), yva).item()
    history.append((loss.item(), val))
    if val < best_val:
        best_val, best_epoch = val, epoch
        best_state = copy.deepcopy(big.state_dict())

print(f"최저 valid {best_val:.3f} @ epoch {best_epoch} / 마지막: train {history[-1][0]:.3f}, valid {history[-1][1]:.3f}")
```
</details>

In [ ]:
for e in [0, 10, 25, 50, 100, 200, 299]:
    print(f"epoch {e:3d}  train {history[e][0]:.3f}  valid {history[e][1]:.3f}")
check("train 손실은 거의 0까지 (외움)", lambda: history[-1][0] < 0.05)
check("valid 최저점이 초반에 있다", lambda: 0 <= best_epoch < 100)
check("마지막 valid가 최저보다 훨씬 나쁘다 = 과적합", lambda: history[-1][1] > best_val * 2)
check("best_val 기록이 맞다", lambda: abs(best_val - min(v for _, v in history)) < 1e-9)
big.load_state_dict(best_state)    # 조기 종료: 가장 좋았던 가중치로 되돌리기

In [ ]:
# 과적합 대책이 아닌 것 하나를 골라요
not_a_fix = ___   # "데이터 늘리기" / "Dropout" / "가중치 감쇠(weight decay)" / "조기 종료" / "에폭 늘리기"

<details><summary>💡 정답 보기 (먼저 직접 풀어보고 열기)</summary>

```python
not_a_fix = "에폭 늘리기"
```
</details>

In [ ]:
check("과적합 대책", lambda: not_a_fix == "에폭 늘리기")

## 4-8. Dataset / DataLoader
큰 데이터는 **배치로 나누고 섞어서** 공급해요. `__len__`과 `__getitem__`만 만들면 돼요.

In [ ]:
from torch.utils.data import Dataset, DataLoader

class TicketDataset(Dataset):
    def __init__(self, X, y):
        self.X, self.y = X, y
    def __len__(self):
        ___
    def __getitem__(self, i):
        ___                             # (특성, 라벨) 튜플

ds = TicketDataset(X, y)                # 위의 200개
loader = DataLoader(ds, batch_size=32, shuffle=True)
batch_sizes = [xb.shape[0] for xb, yb in loader]
batch_sizes

<details><summary>💡 정답 보기 (먼저 직접 풀어보고 열기)</summary>

```python
from torch.utils.data import Dataset, DataLoader

class TicketDataset(Dataset):
    def __init__(self, X, y):
        self.X, self.y = X, y
    def __len__(self):
        return len(self.X)
    def __getitem__(self, i):
        return self.X[i], self.y[i]

ds = TicketDataset(X, y)
loader = DataLoader(ds, batch_size=32, shuffle=True)
batch_sizes = [xb.shape[0] for xb, yb in loader]
batch_sizes
```
</details>

In [ ]:
check("len = 200", lambda: len(ds) == 200)
check("배치 7개 (32×6 + 8)", lambda: batch_sizes == [32] * 6 + [8])
xb, yb = next(iter(loader))
check("배치 모양", lambda: xb.shape == (32, 4) and yb.shape == (32,))

## 4-9. 장치 이동 · 저장/불러오기 · train/eval
- 모델과 데이터는 **같은 장치**에 있어야 해요
- 저장은 `state_dict()`(가중치 dict)로
- `Dropout`은 train 모드에서만 무작위로 꺼요 → 평가 땐 꼭 `eval()`

In [ ]:
import os, tempfile
path = os.path.join(tempfile.mkdtemp(), "m.pt")

model = model.to(device)
Xd = ___                                  # 데이터도 같은 장치로
with torch.no_grad():
    out_before = model(Xd).cpu()

___                                       # model의 가중치를 path에 저장

new_model = nn.Sequential(nn.Linear(4, 16), nn.ReLU(), nn.Linear(16, 2)).to(device)
___                                       # path에서 불러와 new_model에 넣기
new_model.eval()
with torch.no_grad():
    out_after = new_model(Xd).cpu()

<details><summary>💡 정답 보기 (먼저 직접 풀어보고 열기)</summary>

```python
import os, tempfile
path = os.path.join(tempfile.mkdtemp(), "m.pt")

model = model.to(device)
Xd = X.to(device)
with torch.no_grad():
    out_before = model(Xd).cpu()

torch.save(model.state_dict(), path)

new_model = nn.Sequential(nn.Linear(4, 16), nn.ReLU(), nn.Linear(16, 2)).to(device)
new_model.load_state_dict(torch.load(path))
new_model.eval()
with torch.no_grad():
    out_after = new_model(Xd).cpu()
```
</details>

In [ ]:
check("불러온 모델 출력이 같다", lambda: torch.allclose(out_before, out_after, atol=1e-5))

drop = nn.Sequential(nn.Linear(4, 64), nn.Dropout(0.5), nn.Linear(64, 2))
drop.train(); a, b_ = drop(X), drop(X)
drop.eval();  c, d = drop(X), drop(X)
check("train 모드: 같은 입력이어도 매번 다름", lambda: not torch.allclose(a, b_))
check("eval 모드: 항상 같음", lambda: torch.allclose(c, d))

## 4-10. Self-Attention 직접 만들기 (Transformer의 핵심)
`Attention(Q, K, V) = softmax(Q Kᵀ / √d) V`
모든 단어가 다른 모든 단어와의 **관련도**를 한 번에(행렬곱으로) 계산해서 병렬 처리가 돼요.

In [ ]:
torch.manual_seed(0)
T, d = 5, 8                                    # 단어 5개, 차원 8
Q, K, V = torch.randn(T, d), torch.randn(T, d), torch.randn(T, d)

scores = ___                                   # (T, T) 관련도, √d 로 나누기
weights = ___                                  # 행마다 softmax
attn_out = ___                                 # (T, d)

<details><summary>💡 정답 보기 (먼저 직접 풀어보고 열기)</summary>

```python
torch.manual_seed(0)
T, d = 5, 8
Q, K, V = torch.randn(T, d), torch.randn(T, d), torch.randn(T, d)

scores = Q @ K.T / math.sqrt(d)
weights = scores.softmax(dim=-1)
attn_out = weights @ V
```
</details>

In [ ]:
check("관련도 행렬 (T, T)", lambda: scores.shape == (T, T))
check("가중치 행 합 = 1", lambda: torch.allclose(weights.sum(-1), torch.ones(T)))
check("PyTorch 내장과 같다", lambda: torch.allclose(attn_out, F.scaled_dot_product_attention(Q[None], K[None], V[None])[0], atol=1e-5))

## 4-11. 모델 구조 · TensorFlow 대응
TensorFlow는 설치하지 않아도 돼요. 아래 Keras 코드를 읽고 PyTorch와 짝을 맞춰요.

```python
model = tf.keras.Sequential([tf.keras.layers.Dense(16, activation="relu"), tf.keras.layers.Dense(2)])
model.compile(optimizer="adam",
              loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True), metrics=["accuracy"])
model.fit(X, y, epochs=100, batch_size=32, validation_split=0.2)   # 학습 루프를 fit이 대신 해 줌
```

In [ ]:
arch = {
    "작은 필터로 이미지의 지역 패턴(가장자리·모양)을 찾음":           ___,   # "CNN" / "RNN" / "Transformer"
    "순서대로 하나씩 읽으며 기억, 길면 앞을 잊고 병렬 처리가 안 됨":    ___,
    "모든 단어가 모든 단어를 한 번에 봄, 병렬 처리 → 큰 모델 학습":    ___,
}
torch_to_keras = {
    "nn.Linear":            ___,
    "nn.CrossEntropyLoss":  ___,     # 정수 라벨 + logits
    "opt.zero_grad/backward/step 루프": ___,
}
split = {
    "가중치를 학습":                    ___,   # "train" / "valid" / "test"
    "하이퍼파라미터·조기 종료 시점 고르기": ___,
    "마지막에 딱 한 번만 성능 확인":       ___,
}

<details><summary>💡 정답 보기 (먼저 직접 풀어보고 열기)</summary>

```python
arch = {
    "작은 필터로 이미지의 지역 패턴(가장자리·모양)을 찾음":           "CNN",
    "순서대로 하나씩 읽으며 기억, 길면 앞을 잊고 병렬 처리가 안 됨":    "RNN",
    "모든 단어가 모든 단어를 한 번에 봄, 병렬 처리 → 큰 모델 학습":    "Transformer",
}
torch_to_keras = {
    "nn.Linear":            "Dense",
    "nn.CrossEntropyLoss":  "SparseCategoricalCrossentropy",
    "opt.zero_grad/backward/step 루프": "fit",
}
split = {
    "가중치를 학습":                    "train",
    "하이퍼파라미터·조기 종료 시점 고르기": "valid",
    "마지막에 딱 한 번만 성능 확인":       "test",
}
```
</details>

In [ ]:
check("모델 구조", lambda: list(arch.values()) == ["CNN", "RNN", "Transformer"])
check("PyTorch ↔ Keras", lambda: list(torch_to_keras.values()) == ["Dense", "SparseCategoricalCrossentropy", "fit"])
check("데이터 나누기", lambda: list(split.values()) == ["train", "valid", "test"])

In [ ]:
# GPU 메모리 부족(OOM)일 때 해볼 것이 아닌 것 하나
not_oom_fix = ___   # "배치 줄이기" / "혼합 정밀도(fp16/bf16)" / "기울기 누적" / "더 작은 모델" / "학습률 올리기"

<details><summary>💡 정답 보기 (먼저 직접 풀어보고 열기)</summary>

```python
not_oom_fix = "학습률 올리기"
```
</details>

In [ ]:
check("OOM 대책", lambda: not_oom_fix == "학습률 올리기")

## ✍️ 말로 설명해 보기 (30초)
1. PyTorch 학습 루프 5단계를 순서대로 쓰고, `zero_grad`를 빼면 어떻게 되나?
2. 과적합을 어떻게 알아채고 어떻게 줄이나?
3. Transformer가 RNN보다 큰 모델 학습에 유리한 이유는?

<details><summary>모범 답안</summary>

1. 순전파 `model(X)` → 손실 `loss_fn(logits, y)` → `opt.zero_grad()` → `loss.backward()` → `opt.step()`. `zero_grad`를 빼면 이전 배치의 기울기가 계속 더해져서 엉뚱한 방향·크기로 갱신돼요.
2. train 손실은 내려가는데 valid 손실이 올라가면 과적합이에요. 데이터 늘리기, Dropout, 가중치 감쇠, 조기 종료(valid 최저 시점 가중치 사용), 더 작은 모델로 줄여요.
3. RNN은 앞 단계 결과가 있어야 다음을 계산해서 병렬화가 안 되고 긴 문장에서 앞을 잊어요. Transformer는 Self-Attention으로 모든 단어 쌍의 관련도를 행렬곱 한 번에 계산해 GPU로 병렬 처리되고, 먼 단어도 바로 연결돼서 큰 모델·큰 데이터 학습에 유리해요.
</details>